# Session 3: Phi-3 LoRA, inference and Ragas
Run with Python 3.11 after `uv sync --extra training --extra evaluation --group dev`. CUDA is needed for the training cell. Configuration inspection and ROUGE evaluation can run on CPU. Use the Vertex smoke test when a local GPU is unavailable. Training and inference cells download the Phi-3 weights only when you run them.

In [ ]:
from pathlib import Path
import json, os, sys
os.environ['RAGAS_DO_NOT_TRACK'] = 'true'
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')
import pandas as pd
from agentic_workflows.constants import RAW_DATASET_URI
from agentic_workflows.data_processing import format_yoda_dataset
from agentic_workflows.inference import download_model, build_prompt, generate_response, extract_response
from agentic_workflows.evaluation import compute_metrics, aggregate_metrics


## Inspect configurations and the training data
This uses the pinned TRL API required by the course. The validation split will come from the training CSV; the held-out test CSV remains separate.

In [ ]:
import torch
from peft import LoraConfig
from transformers import BitsAndBytesConfig
from trl import SFTConfig
hyperparameters = dict(learning_rate=3e-4, num_train_epochs=3.0, lora_rank=8, lora_alpha=16, max_seq_length=128, batch_size=4, gradient_accumulation_steps=4, validation_size=0.1, seed=42)
lora_config = LoraConfig(r=hyperparameters['lora_rank'], lora_alpha=hyperparameters['lora_alpha'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['o_proj', 'qkv_proj', 'gate_up_proj', 'down_proj'])
quantization_config = None
if sys.platform != 'darwin':
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
sft_config = SFTConfig(output_dir=str(ROOT / 'models/checkpoints'), logging_dir=str(ROOT / 'logs'), report_to=['tensorboard'], use_cpu=not torch.cuda.is_available(), fp16=torch.cuda.is_available(), bf16=False, max_seq_length=128, eval_strategy='epoch', learning_rate=hyperparameters['learning_rate'], num_train_epochs=hyperparameters['num_train_epochs'], per_device_train_batch_size=hyperparameters['batch_size'], gradient_accumulation_steps=hyperparameters['gradient_accumulation_steps'])
print('CUDA available:', torch.cuda.is_available())
display(hyperparameters, lora_config, quantization_config, sft_config)


In [ ]:
from datasets import Dataset as HFDataset
train_path, test_path = format_yoda_dataset(RAW_DATASET_URI, str(ROOT / 'data/processed/train.csv'), str(ROOT / 'data/processed/test.csv'))
train_frame = pd.read_csv(train_path)
conversations = [json.loads(value) for value in train_frame['messages']]
dataset = HFDataset.from_list([{'messages': messages} for messages in conversations])
validation_split = dataset.train_test_split(test_size=hyperparameters['validation_size'], seed=hyperparameters['seed'])
display(dataset[0], validation_split)


## Local GPU smoke training
The component prints the trainable parameter fraction and LoRA module names, runs two optimizer steps, and saves the adapters, tokenizer, history and TensorBoard events. Remove `max_steps=2` for full training. If CUDA is unavailable, submit `uv run python scripts/pipeline_runner.py --smoke-test` from the project directory.

In [ ]:
from kfp.dsl import Dataset, Model, Metrics
from agentic_workflows.pipeline_components.fine_tuning_component import fine_tuning_component
if not torch.cuda.is_available():
    raise RuntimeError('Use the Vertex AI smoke test: local fine-tuning requires CUDA.')
training_input = Dataset()
training_input.path = train_path
adapter_artifact = Model()
adapter_artifact.path = str(ROOT / 'models/phi3-yoda-adapter')
training_metrics = Metrics()
training_metrics.path = str(ROOT / 'logs')
fine_tuning_component.python_func(training_input, adapter_artifact, training_metrics, max_steps=2, **hyperparameters)
display(training_metrics.metadata)


## Download the cloud artifact (when training on Vertex)
Copy the model artifact URI from the Vertex pipeline console. Leave it empty to use the locally saved adapter. All files are downloaded recursively.

In [ ]:
MODEL_URI = ''  # gs://YOUR_BUCKET/.../model
adapter_dir = ROOT / 'models/phi3-yoda-adapter'
if MODEL_URI:
    download_model(MODEL_URI, str(adapter_dir))
assert (adapter_dir / 'adapter_config.json').exists(), 'Run training or download the model artifact first.'


## Predict three held-out rows
The artifact contains LoRA adapters. Load the base model referenced by `adapter_config.json`, then attach the adapters with PEFT. For cloud training, set `TEST_DATASET_URI` to the test artifact URI if local test data is unavailable.

In [ ]:
from peft import PeftConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc
# Free any local training model before loading the inference copy.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
adapter_config = PeftConfig.from_pretrained(str(adapter_dir))
base = AutoModelForCausalLM.from_pretrained(adapter_config.base_model_name_or_path, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map='auto', attn_implementation='eager')
model = PeftModel.from_pretrained(base, str(adapter_dir))
model.config.use_cache = True
tokenizer = AutoTokenizer.from_pretrained(str(adapter_dir))
TEST_DATASET_URI = str(ROOT / 'data/processed/test.csv')
test_frame = pd.read_csv(TEST_DATASET_URI).head(3)
rows = []
for value in test_frame['messages']:
    messages = json.loads(value)
    sentence = next(message['content'] for message in messages if message['role'] == 'user')
    reference = next(message['content'] for message in messages if message['role'] == 'assistant')
    response = extract_response(generate_response(model, tokenizer, build_prompt(tokenizer, sentence)))
    rows.append(dict(user_input=sentence, reference=reference, extracted_response=response))
predictions = pd.DataFrame(rows)
prediction_path = ROOT / 'data/predictions.csv'
prediction_path.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(prediction_path, index=False)
display(predictions)


## Evaluate without an LLM
This section can run independently using a predictions CSV downloaded from the pipeline. ROUGE F1 values range from zero to one. Lexical overlap is a useful check, but does not fully measure how much the output sounds like Yoda.

In [ ]:
from ragas.metrics import RougeScore
PREDICTIONS_URI = str(ROOT / 'data/predictions.csv')
predictions = pd.read_csv(PREDICTIONS_URI, keep_default_na=False)
definitions = [RougeScore(rouge_type=kind, name=kind) for kind in ['rouge1', 'rouge2', 'rougeL']]
scores = pd.DataFrame([compute_metrics(row.user_input, row.extracted_response, row.reference, definitions) for row in predictions.itertuples(index=False)])
display(pd.concat([predictions, scores], axis=1), aggregate_metrics(scores))
scores.to_csv(ROOT / 'data/evaluation_results.csv', index=False)
